In [0]:
# Databricks notebook source

# 02 - Silver Transformation

Lê os dados brutos de um mês (`year_month`) da camada Bronze, aplica regras de qualidade e tipagem,
e grava o resultado limpo na camada Silver.

Linhas que falham em qualquer regra de qualidade **não são descartadas** — vão para uma tabela de
quarentena (`trips_silver_rejected`), com o motivo da rejeição registrado. Isso preserva
observabilidade: dá pra medir quanto % dos dados de um mês é sujo e por quê, em vez de só sumir com
eles silenciosamente.

A carga é feita por **overwrite de partição** (`replaceWhere`): cada execução reprocessa o mês
inteiro e substitui o que existia antes para aquele `year_month` — diferente do MERGE por hash usado
no Bronze, mas igualmente idempotente.

## Configuração e widgets

In [0]:
dbutils.widgets.text("year", "2024", "Ano (YYYY)")
dbutils.widgets.text("month", "01", "Mês (MM)")
 
year = dbutils.widgets.get("year")
month = dbutils.widgets.get("month")
year_month = f"{year}-{month}"
 
CATALOG = "nyc_taxi"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
 
BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.trips_bronze"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.trips_silver"
SILVER_REJECTED_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.trips_silver_rejected"
 
print(f"Processando mês: {year_month}")

## Criação das tabelas Silver (idempotente)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
    trip_id STRING,
    year_month STRING,
    vendor_id INT,
    pickup_datetime TIMESTAMP,
    dropoff_datetime TIMESTAMP,
    trip_duration_minutes DOUBLE,
    pickup_hour INT,
    pickup_dow INT,
    passenger_count DOUBLE,
    trip_distance DOUBLE,
    rate_code_id DOUBLE,
    store_and_fwd_flag STRING,
    pu_location_id INT,
    do_location_id INT,
    payment_type BIGINT,
    fare_amount DOUBLE,
    extra DOUBLE,
    mta_tax DOUBLE,
    tip_amount DOUBLE,
    tolls_amount DOUBLE,
    improvement_surcharge DOUBLE,
    total_amount DOUBLE,
    congestion_surcharge DOUBLE,
    airport_fee DOUBLE,
    _silver_processed_at TIMESTAMP
)
USING DELTA
PARTITIONED BY (year_month)
""")
 
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_REJECTED_TABLE} (
    trip_id STRING,
    year_month STRING,
    rejection_reason STRING,
    raw_pickup_datetime TIMESTAMP,
    raw_dropoff_datetime TIMESTAMP,
    raw_trip_distance DOUBLE,
    raw_passenger_count DOUBLE,
    raw_fare_amount DOUBLE,
    raw_pu_location_id INT,
    raw_do_location_id INT,
    _rejected_at TIMESTAMP
)
USING DELTA
PARTITIONED BY (year_month)
""")
 
print("Tabelas Silver prontas.")

## Leitura do mês na camada Bronze

In [0]:
from pyspark.sql import functions as F
 
bronze_df = spark.table(BRONZE_TABLE).filter(F.col("year_month") == year_month)
 
bronze_row_count = bronze_df.count()
print(f"Linhas lidas do Bronze para {year_month}: {bronze_row_count}")
 
if bronze_row_count == 0:
    dbutils.notebook.exit(f"Nenhuma linha encontrada no Bronze para {year_month}. Rode a ingestão primeiro.")

## Colunas derivadas

Calculadas antes das regras de qualidade, porque algumas regras dependem delas (ex: duração da
corrida).

In [0]:
enriched_df = (
    bronze_df
    .withColumn(
        "trip_duration_minutes",
        (F.col("tpep_dropoff_datetime").cast("long") - F.col("tpep_pickup_datetime").cast("long")) / 60.0,
    )
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_dow", F.dayofweek("tpep_pickup_datetime"))
)

## Regras de qualidade

Cada regra é uma condição que a linha **precisa satisfazer para ser válida**. Uma linha pode falhar
em mais de uma regra ao mesmo tempo — todos os motivos são registrados, não só o primeiro.
 

In [0]:
quality_rules = [
    ("trip_distance_invalida", (F.col("trip_distance") > 0) & (F.col("trip_distance") <= 100)),
    ("passenger_count_invalido", (F.col("passenger_count") >= 1) & (F.col("passenger_count") <= 8)),
    ("fare_amount_negativo", F.col("fare_amount") >= 0),
    ("total_amount_negativo", F.col("total_amount") >= 0),
    ("datas_nulas_ou_invertidas", F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime")),
    ("duracao_fora_do_intervalo", (F.col("trip_duration_minutes") >= 1) & (F.col("trip_duration_minutes") <= 1440)),
    ("pu_location_id_invalido", (F.col("PULocationID") >= 1) & (F.col("PULocationID") <= 263)),
    ("do_location_id_invalido", (F.col("DOLocationID") >= 1) & (F.col("DOLocationID") <= 263)),
    ("pickup_fora_do_mes_esperado", F.date_format("tpep_pickup_datetime", "yyyy-MM") == year_month),
]
 
reason_columns = [
    F.when(~condition, F.lit(rule_name)) for rule_name, condition in quality_rules
]
 
flagged_df = enriched_df.withColumn("rejection_reasons_raw", F.array(*reason_columns))
flagged_df = flagged_df.withColumn(
    "rejection_reasons", F.expr("filter(rejection_reasons_raw, x -> x is not null)")
)

## Split entre válidos e rejeitados

In [0]:
valid_df = flagged_df.filter(F.size("rejection_reasons") == 0)
rejected_df = flagged_df.filter(F.size("rejection_reasons") > 0)
 
valid_count = valid_df.count()
rejected_count = rejected_df.count()
rejection_rate = round((rejected_count / bronze_row_count) * 100, 2) if bronze_row_count > 0 else 0
 
print(f"Válidas: {valid_count}")
print(f"Rejeitadas: {rejected_count}")
print(f"Taxa de rejeição: {rejection_rate}%")

### Detalhamento dos motivos de rejeição

Uma linha rejeitada por mais de um motivo conta em cada um — a soma pode passar do total de
rejeitados.

In [0]:
rejection_breakdown = (
    rejected_df
    .withColumn("reason", F.explode("rejection_reasons"))
    .groupBy("reason")
    .count()
    .orderBy(F.desc("count"))
)
 
display(rejection_breakdown)

## Preparação final dos válidos (renomeando para o schema Silver)

In [0]:
final_valid_df = valid_df.select(
    F.sha2(F.concat_ws("||", F.col("row_hash"), F.lit("silver")), 256).alias("trip_id"),
    F.col("year_month"),
    F.col("VendorID").alias("vendor_id"),
    F.col("tpep_pickup_datetime").alias("pickup_datetime"),
    F.col("tpep_dropoff_datetime").alias("dropoff_datetime"),
    F.col("trip_duration_minutes"),
    F.col("pickup_hour"),
    F.col("pickup_dow"),
    F.col("passenger_count"),
    F.col("trip_distance"),
    F.col("RatecodeID").alias("rate_code_id"),
    F.col("store_and_fwd_flag"),
    F.col("PULocationID").alias("pu_location_id"),
    F.col("DOLocationID").alias("do_location_id"),
    F.col("payment_type"),
    F.col("fare_amount"),
    F.col("extra"),
    F.col("mta_tax"),
    F.col("tip_amount"),
    F.col("tolls_amount"),
    F.col("improvement_surcharge"),
    F.col("total_amount"),
    F.col("congestion_surcharge"),
    F.col("airport_fee"),
    F.current_timestamp().alias("_silver_processed_at"),
)
 
final_rejected_df = rejected_df.select(
    F.sha2(F.concat_ws("||", F.col("row_hash"), F.lit("silver")), 256).alias("trip_id"),
    F.col("year_month"),
    F.concat_ws("; ", F.col("rejection_reasons")).alias("rejection_reason"),
    F.col("tpep_pickup_datetime").alias("raw_pickup_datetime"),
    F.col("tpep_dropoff_datetime").alias("raw_dropoff_datetime"),
    F.col("trip_distance").alias("raw_trip_distance"),
    F.col("passenger_count").alias("raw_passenger_count"),
    F.col("fare_amount").alias("raw_fare_amount"),
    F.col("PULocationID").alias("raw_pu_location_id"),
    F.col("DOLocationID").alias("raw_do_location_id"),
    F.current_timestamp().alias("_rejected_at"),
)

## Gravação (overwrite por partição)
`replaceWhere` diz ao Delta: "apague só as linhas que batem com esse predicado, e insira essas no
lugar" — como um `DELETE WHERE year_month = '...'` seguido de `INSERT`, só que atômico em uma única
operação.

In [0]:
(
    final_valid_df.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"year_month = '{year_month}'")
    .saveAsTable(SILVER_TABLE)
)
 
(
    final_rejected_df.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"year_month = '{year_month}'")
    .saveAsTable(SILVER_REJECTED_TABLE)
)
 
print(f"Silver atualizada para {year_month}: {valid_count} válidas, {rejected_count} rejeitadas.")

## Verificação rápida

In [0]:
display(
    spark.table(SILVER_TABLE)
    .filter(F.col("year_month") == year_month)
    .limit(10)
)